# SQL Practice - Day 5

15 Interview-Level SQL Questions

Topics: Date Functions, LAG, LEAD, CTE, Window Functions, Correlated Subqueries, Joins, Aggregation.

## Q1

**Task:** Find each customer's first and latest order date.

In [1]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'customer_id':[101,101,102,103,102,101],'order_date':['2025-01-01','2025-01-15','2025-01-05','2025-01-10','2025-02-20','2025-03-01']})
df

,order_id,customer_id,order_date
0,1,101,2025-01-01
1,2,101,2025-01-15
2,3,102,2025-01-05
3,4,103,2025-01-10
4,5,102,2025-02-20
5,6,101,2025-03-01


In [14]:
import pandasql
from pandasql import sqldf

sqldf("""
select customer_id, first_order , last_order
from (select customer_id , max(order_date) as last_order,
                min(order_date) as first_order
                  from df
      group by customer_id) t





""")

,customer_id,first_order,last_order
0,101,2025-01-01,2025-03-01
1,102,2025-01-05,2025-02-20
2,103,2025-01-10,2025-01-10


## Q2

**Task:** Find customers who didn't order in February 2025.

In [15]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'customer_id':[101,101,102,103,102,101],'order_date':['2025-01-01','2025-01-15','2025-01-05','2025-02-10','2025-02-20','2025-03-01']})
df

,order_id,customer_id,order_date
0,1,101,2025-01-01
1,2,101,2025-01-15
2,3,102,2025-01-05
3,4,103,2025-02-10
4,5,102,2025-02-20
5,6,101,2025-03-01


In [19]:
sqldf("""
SELECT DISTINCT customer_id
FROM df
WHERE customer_id NOT IN (
    SELECT customer_id
    FROM df
    WHERE order_date >= '2025-02-01'
      AND order_date < '2025-03-01'
);

""")

,customer_id
0,101


## Q3

**Task:** Show previous order amount using LAG().

In [ ]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,101,102,102],'order_date':['2025-01-01','2025-01-05','2025-01-10','2025-01-03','2025-01-08'],'amount':[100,200,150,300,250]})
df

In [ ]:
# Write your SQL here

## Q4

**Task:** Running total by customer.

In [ ]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,101,102,102],'order_date':['2025-01-01','2025-01-05','2025-01-10','2025-01-03','2025-01-08'],'amount':[100,200,150,300,250]})
df

In [ ]:
# Write your SQL here

## Q5

**Task:** Employees hired in the same month but different years.

In [ ]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4],'name':['A','B','C','D'],'hire_date':['2022-01-10','2023-01-20','2022-02-15','2024-02-28']})
df

In [ ]:
# Write your SQL here

## Q6

**Task:** Customers above average customer spending using CTE.

In [24]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,102,101],'amount':[100,200,300,500,200,400]})
df

,customer_id,amount
0,101,100
1,101,200
2,102,300
3,103,500
4,102,200
5,101,400


In [25]:
sqldf("""
with my_cte as (
select customer_id , sum(amount) as total_spending
from df
group by customer_id 
having sum(amount) > (select avg(amount) 
                        from df)
)
select * from my_cte
""")

,customer_id,total_spending
0,101,700
1,102,500
2,103,500


## Q7

**Task:** Top 2 products by sales in each category.

In [26]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4,5,6],'category':['A','A','A','B','B','B'],'sales':[100,300,200,500,400,450]})
df

,product_id,category,sales
0,1,A,100
1,2,A,300
2,3,A,200
3,4,B,500
4,5,B,400
5,6,B,450


In [34]:
sqldf("""
select  category,product_id  
from (select product_id , category,
      dense_rank()
      over(partition by category order by sales desc) as rn
    from df) t
where rn <= 2



""")

,category,product_id
0,A,2
1,A,3
2,B,4
3,B,6


## Q8

**Task:** Departments above overall average salary.

In [40]:
import pandas as pd
df=pd.DataFrame({'department':['HR','HR','IT','IT','Sales','Sales'],'salary':[50000,60000,70000,80000,90000,85000]})
df

,department,salary
0,HR,50000
1,HR,60000
2,IT,70000
3,IT,80000
4,Sales,90000
5,Sales,85000


In [42]:
sqldf("""
select department
from df
group by department
having avg(salary) > (
    select avg(salary)
    from df
);



""")

,department
0,IT
1,Sales


## Q9

**Task:** Latest order using ROW_NUMBER().

In [50]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5],'customer_id':[101,101,102,102,103],'order_date':['2025-01-01','2025-02-01','2025-01-10','2025-03-01','2025-02-15'],'amount':[100,200,300,250,500]})
df

,order_id,customer_id,order_date,amount
0,1,101,2025-01-01,100
1,2,101,2025-02-01,200
2,3,102,2025-01-10,300
3,4,102,2025-03-01,250
4,5,103,2025-02-15,500


In [51]:
sqldf("""
    select customer_id , order_date, rn
    from (select customer_id , order_date ,
           row_number()
          over(order by order_date desc) as rn
          from df) t
    where rn = 1
    
""")

,customer_id,order_date,rn
0,102,2025-03-01,1


## Q10

**Task:** Customers with more than one order on same day.

In [52]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5],'customer_id':[101,101,102,102,102],'order_date':['2025-01-01','2025-01-01','2025-02-01','2025-02-01','2025-03-01']})
df

,order_id,customer_id,order_date
0,1,101,2025-01-01
1,2,101,2025-01-01
2,3,102,2025-02-01
3,4,102,2025-02-01
4,5,102,2025-03-01


In [54]:
sqldf("""
select customer_id , order_date , count(*) as count
from df
group by customer_id , order_date
having count(*) > 1

""")

,customer_id,order_date,count
0,101,2025-01-01,2
1,102,2025-02-01,2


## Q11

**Task:** Next order date using LEAD().

In [55]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,102],'order_date':['2025-01-01','2025-02-01','2025-01-15','2025-03-01']})
df

,customer_id,order_date
0,101,2025-01-01
1,101,2025-02-01
2,102,2025-01-15
3,102,2025-03-01


In [ ]:
# Write your SQL here

## Q12

**Task:** Second highest spending customer using CTE + DENSE_RANK().

In [56]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,104],'amount':[100,300,500,700,600]})
df

,customer_id,amount
0,101,100
1,101,300
2,102,500
3,103,700
4,104,600


In [78]:
sqldf("""
    with my_cte as (
    select customer_id , sum(amount) as total_spending,
    dense_rank()
    over(order by sum(amount) desc) as rn
    from df
    group by customer_id
    )
    select * from my_cte
    where rn = 2


""")

,customer_id,total_spending,rn
0,104,600,2


## Q13

**Task:** Products above category average.

In [85]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4,5,6],'category':['A','A','A','B','B','B'],'price':[100,150,200,300,250,350]})
df

,product_id,category,price
0,1,A,100
1,2,A,150
2,3,A,200
3,4,B,300
4,5,B,250
5,6,B,350


In [87]:
sqldf("""
    select *
    from df a
    where price > (select avg(price) as avg_price
                    from df b
                    where a.category = b.category)



""")

,product_id,category,price
0,3,A,200
1,6,B,350


## Q14

**Task:** Customers with no orders.

In [88]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,102,103,104],'name':['A','B','C','D']})
df1=pd.DataFrame({'order_id':[1,2,3],'customer_id':[101,102,102]})
print(df)
print(df1)

   customer_id name
0          101    A
1          102    B
2          103    C
3          104    D
   order_id  customer_id
0         1          101
1         2          102
2         3          102


In [94]:
sqldf("""
        select c.name ,o.order_id 
        from df c
        left join df1 o on c.customer_id = o.customer_id
        where o.order_id is null
                


""")

,name,order_id
0,C,None
1,D,None


## Q15

**Task:** Highest paid employee per department using CTE + ROW_NUMBER().

In [95]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D','E','F'],'department':['HR','HR','IT','IT','Sales','Sales'],'salary':[50000,60000,80000,70000,90000,85000]})
df

,name,department,salary
0,A,HR,50000
1,B,HR,60000
2,C,IT,80000
3,D,IT,70000
4,E,Sales,90000
5,F,Sales,85000


In [101]:
sqldf("""
with my_cte as (
    select name , department ,salary , rn
    from ( select name , department , salary,
    row_number()
    over(partition by department order by salary desc) as rn
    from df) t
    )
    select * from my_cte
    where rn = 1



""")

,name,department,salary,rn
0,B,HR,60000,1
1,C,IT,80000,1
2,E,Sales,90000,1
